# Medical MNIST - Comprehensive Debugging & Analysis

This notebook provides advanced debugging, error analysis, and medical-specific insights for the Medical MNIST classification pipeline.

**Features:**
- Data quality debugging
- Wrong prediction analysis
- Grad-CAM visualization for model interpretability
- Medical-specific metrics and analysis
- Per-class performance analysis
- Confusion matrix deep dive

In [5]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
import cv2
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.manifold import TSNE
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('../src')

# Import project modules
from config.config import Config
from config.paths import Paths
from src.data_loader import load_medical_mnist_data, create_data_loaders, check_data_exists
from src.modeling.models import get_model
from src.training.trainer import CVTrainer
from src.utils import set_seed, get_device

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Setup complete!")

Setup complete!


## 2. Load Data and Models

In [6]:
# Initialize config
config = Config()
config.device = get_device("auto")
set_seed(config.seed)

# Load data
print("Loading Medical MNIST data...")
image_paths, labels, class_names = load_medical_mnist_data(Paths.RAW_DATA_DIR)
print(f"Loaded {len(image_paths)} images across {len(class_names)} classes")
print(f"Classes: {class_names}")

# Split data
from src.data_loader import split_train_test
train_paths, train_labels, test_paths, test_labels = split_train_test(
    image_paths, labels, test_size=0.2, random_state=config.seed
)

print(f"Train: {len(train_paths)}, Test: {len(test_paths)}")

# Create test data loader for analysis
_, _, test_loader = create_data_loaders(
    train_paths[:1], [0], None, None, test_paths, test_labels, config
)

Loading Medical MNIST data...
Loaded 58954 images across 6 classes
Classes: ['AbdomenCT', 'BreastMRI', 'CXR', 'ChestCT', 'Hand', 'HeadCT']
Train: 47163, Test: 11791


In [7]:
# Load trained models
def load_trained_models(model_name="cnn", n_folds=3):
    """Load trained models from outputs"""
    models = []
    for fold in range(n_folds):
        model_path = Paths.MODELS_DIR / f"model_fold_{fold}.pth"
        if model_path.exists():
            model = get_model(model_name, config)
            model.load_state_dict(torch.load(model_path, map_location=config.device))
            model.eval()
            models.append(model)
            print(f"Loaded fold {fold} model from {model_path}")
    return models

trained_models = load_trained_models("cnn", 3)
print(f"Loaded {len(trained_models)} trained models")

RuntimeError: Error(s) in loading state_dict for SimpleCNN:
	Missing key(s) in state_dict: "features.0.weight", "features.0.bias", "features.1.weight", "features.1.bias", "features.1.running_mean", "features.1.running_var", "features.3.weight", "features.3.bias", "features.4.weight", "features.4.bias", "features.4.running_mean", "features.4.running_var", "features.8.weight", "features.8.bias", "features.9.weight", "features.9.bias", "features.9.running_mean", "features.9.running_var", "features.11.weight", "features.11.bias", "features.12.weight", "features.12.bias", "features.12.running_mean", "features.12.running_var", "features.16.weight", "features.16.bias", "features.17.weight", "features.17.bias", "features.17.running_mean", "features.17.running_var", "classifier.0.weight", "classifier.0.bias", "classifier.1.weight", "classifier.1.bias", "classifier.1.running_mean", "classifier.1.running_var", "classifier.4.weight", "classifier.4.bias". 
	Unexpected key(s) in state_dict: "model_s

## 3. Medical Data Quality Debugging

In [ ]:
def analyze_data_quality():
    """Comprehensive data quality analysis"""
    print("=" * 60)
    print("MEDICAL DATA QUALITY ANALYSIS")
    print("=" * 60)
    
    # Basic statistics
    print(f"\n📊 Basic Statistics:")
    print(f"  Total images: {len(image_paths)}")
    print(f"  Classes: {len(class_names)}")
    print(f"  Image format distribution:")
    
    # Check image formats
    formats = Counter([Path(p).suffix.lower() for p in image_paths])
    for fmt, count in formats.items():
        print(f"    {fmt}: {count} ({count/len(image_paths)*100:.1f}%)")
    
    # Class distribution
    print(f"\n🏥 Class Distribution:")
    class_counts = Counter(labels)
    for i, (cls_name, count) in enumerate(zip(class_names, [class_counts[i] for i in range(len(class_names))])):
        percentage = count / len(labels) * 100
        print(f"  {cls_name:12}: {count:6} ({percentage:5.1f}%)")
    
    # Check for data imbalance
    max_count = max(class_counts.values())
    min_count = min(class_counts.values())
    imbalance_ratio = max_count / min_count
    print(f"\n⚖️  Data Imbalance:")
    print(f"  Max class: {max_count}")
    print(f"  Min class: {min_count}")
    print(f"  Imbalance ratio: {imbalance_ratio:.2f}:1")
    
    if imbalance_ratio > 1.5:
        print(f"  ⚠️  Moderate imbalance detected")
    elif imbalance_ratio > 2.0:
        print(f"  ❌ High imbalance detected - consider class weighting")
    else:
        print(f"  ✅ Balanced dataset")
    
    # Image quality checks
    print(f"\n🔍 Image Quality Analysis:")
    
    # Sample images for quality analysis
    sample_size = min(100, len(image_paths))
    sample_indices = np.random.choice(len(image_paths), sample_size, replace=False)
    
    sizes = []
    corrupted = 0
    
    for idx in sample_indices:
        try:
            img = Image.open(image_paths[idx])
            sizes.append(img.size)
            
            # Check for corruption
            img.verify()
            img = Image.open(image_paths[idx])  # Reopen after verify
            
        except Exception as e:
            corrupted += 1
    
    # Size statistics
    if sizes:
        widths, heights = zip(*sizes)
        print(f"  Image sizes: {min(widths)}x{min(heights)} to {max(widths)}x{max(heights)}")
        print(f"  Most common size: {Counter(sizes).most_common(1)[0][0]}")
        print(f"  Corrupted images: {corrupted}/{sample_size} ({corrupted/sample_size*100:.1f}%)")
    
    return class_counts, imbalance_ratio

class_counts, imbalance_ratio = analyze_data_quality()

In [ ]:
def visualize_data_distribution():
    """Visualize data distribution and quality"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Class distribution
    ax1 = axes[0, 0]
    class_counts = [Counter(labels)[i] for i in range(len(class_names))]
    bars = ax1.bar(class_names, class_counts, color=sns.color_palette("husl", len(class_names)))
    ax1.set_title('Class Distribution', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Medical Classes')
    ax1.set_ylabel('Number of Images')
    ax1.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar, count in zip(bars, class_counts):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + max(class_counts)*0.01,
                f'{count}', ha='center', va='bottom')
    
    # 2. Sample images from each class
    ax2 = axes[0, 1]
    samples_per_class = 3
    for i, class_name in enumerate(class_names):
        class_indices = [j for j, label in enumerate(labels) if label == i]
        sample_indices = np.random.choice(class_indices, min(samples_per_class, len(class_indices)), replace=False)
        
        for j, idx in enumerate(sample_indices):
            img = Image.open(image_paths[idx])
            img = img.resize((64, 64))
            
            # Calculate subplot position
            row = i * samples_per_class + j
            col = row % len(class_names)
            row_pos = row // len(class_names)
            
            if row_pos < 3:  # Only show first 3 rows
                ax2.imshow(img, cmap='gray', extent=[col, col+1, 3-row_pos-1, 3-row_pos])
                
                # Add class label for first row
                if j == 0:
                    ax2.text(col + 0.5, 3-row_pos-0.5, class_name, 
                            ha='center', va='center', fontsize=8, fontweight='bold')
    
    ax2.set_xlim(0, len(class_names))
    ax2.set_ylim(0, 3)
    ax2.set_title('Sample Images per Class', fontsize=14, fontweight='bold')
    ax2.axis('off')
    
    # 3. Train/Test split distribution
    ax3 = axes[1, 0]
    train_counts = [Counter(train_labels)[i] for i in range(len(class_names))]
    test_counts = [Counter(test_labels)[i] for i in range(len(class_names))]
    
    x = np.arange(len(class_names))
    width = 0.35
    
    ax3.bar(x - width/2, train_counts, width, label='Train', alpha=0.8)
    ax3.bar(x + width/2, test_counts, width, label='Test', alpha=0.8)
    ax3.set_title('Train/Test Split Distribution', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Medical Classes')
    ax3.set_ylabel('Number of Images')
    ax3.set_xticks(x)
    ax3.set_xticklabels(class_names, rotation=45)
    ax3.legend()
    
    # 4. Data imbalance visualization
    ax4 = axes[1, 1]
    percentages = [count/len(labels)*100 for count in class_counts]
    colors = ['red' if p < 15 or p > 20 else 'green' for p in percentages]
    
    bars = ax4.bar(class_names, percentages, color=colors, alpha=0.7)
    ax4.axhline(y=100/len(class_names), color='black', linestyle='--', alpha=0.5, label='Expected (balanced)')
    ax4.set_title('Class Balance Analysis', fontsize=14, fontweight='bold')
    ax4.set_xlabel('Medical Classes')
    ax4.set_ylabel('Percentage (%)')
    ax4.tick_params(axis='x', rotation=45)
    ax4.legend()
    
    # Add percentage labels
    for bar, pct in zip(bars, percentages):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{pct:.1f}%', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

visualize_data_distribution()

# Load trained models
def load_trained_models(model_name="cnn", n_folds=3):
    """Load trained models from outputs"""
    models = []
    for fold in range(n_folds):
        model_path = Paths.MODELS_DIR / f"model_fold_{fold}.pth"
        if model_path.exists():
            model = get_model(model_name, config)
            # Load checkpoint and extract model state dict
            checkpoint = torch.load(model_path, map_location=config.device)
            if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
            else:
                model.load_state_dict(checkpoint)
            model.eval()
            models.append(model)
            print(f"Loaded fold {fold} model from {model_path}")
        else:
            print(f"Model not found: {model_path}")
    return models

trained_models = load_trained_models("cnn", 3)
print(f"Loaded {len(trained_models)} trained models")

In [ ]:
def get_model_predictions(models, data_loader):
    """Get ensemble predictions from multiple models"""
    all_predictions = []
    all_probabilities = []
    all_labels = []
    
    for model in models:
        model.eval()
        model_preds = []
        model_probs = []
        
        with torch.no_grad():
            for batch_idx, (images, labels_batch) in enumerate(data_loader):
                images = images.to(config.device)
                labels_batch = labels_batch.to(config.device)
                
                outputs = model(images)
                probs = F.softmax(outputs, dim=1)
                preds = torch.argmax(probs, dim=1)
                
                model_preds.extend(preds.cpu().numpy())
                model_probs.extend(probs.cpu().numpy())
                
                if batch_idx == 0:  # Store labels once
                    all_labels.extend(labels_batch.cpu().numpy())
        
        all_predictions.append(model_preds)
        all_probabilities.append(model_probs)
    
    # Ensemble predictions (average probabilities)
    avg_probabilities = np.mean(all_probabilities, axis=0)
    ensemble_predictions = np.argmax(avg_probabilities, axis=1)
    
    return ensemble_predictions, avg_probabilities, np.array(all_labels)

# Get predictions
predictions, probabilities, true_labels = get_model_predictions(trained_models, test_loader)

print(f"Generated predictions for {len(predictions)} test samples")
print(f"Overall accuracy: {np.mean(predictions == true_labels)*100:.2f}%")

In [ ]:
def analyze_model_performance():
    """Comprehensive model performance analysis"""
    print("=" * 60)
    print("MODEL PERFORMANCE ANALYSIS")
    print("=" * 60)
    
    # Overall metrics
    accuracy = np.mean(predictions == true_labels)
    print(f"\n📈 Overall Performance:")
    print(f"  Accuracy: {accuracy*100:.2f}%")
    print(f"  Correct predictions: {np.sum(predictions == true_labels)}/{len(predictions)}")
    print(f"  Wrong predictions: {np.sum(predictions != true_labels)}/{len(predictions)}")
    
    # Per-class performance
    print(f"\n🏥 Per-Class Performance:")
    report = classification_report(true_labels, predictions, target_names=class_names, 
                                   labels=list(range(len(class_names))), output_dict=True)
    
    for i, class_name in enumerate(class_names):
        class_report = report[class_name]
        print(f"  {class_name:12}: Precision={class_report['precision']:.3f}, "
              f"Recall={class_report['recall']:.3f}, F1={class_report['f1-score']:.3f}")
    
    # Identify worst performing classes
    f1_scores = [report[class_name]['f1-score'] for class_name in class_names]
    worst_class_idx = np.argmin(f1_scores)
    best_class_idx = np.argmax(f1_scores)
    
    print(f"\n🏆 Best performing class: {class_names[best_class_idx]} (F1={f1_scores[best_class_idx]:.3f})")
    print(f"⚠️  Worst performing class: {class_names[worst_class_idx]} (F1={f1_scores[worst_class_idx]:.3f})")
    
    return report, f1_scores

performance_report, f1_scores = analyze_model_performance()

In [ ]:
def plot_confusion_matrix_detailed():
    """Detailed confusion matrix visualization"""
    cm = confusion_matrix(true_labels, predictions)
    
    # Normalize confusion matrix
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, ax=ax1)
    ax1.set_title('Confusion Matrix (Raw Counts)', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Predicted Label')
    ax1.set_ylabel('True Label')
    
    # Normalized
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, ax=ax2)
    ax2.set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Predicted Label')
    ax2.set_ylabel('True Label')
    
    plt.tight_layout()
    plt.show()
    
    # Print confusion analysis
    print("\n🔍 Confusion Matrix Analysis:")
    for i, true_class in enumerate(class_names):
        correct = cm[i, i]
        total = cm[i, :].sum()
        
        # Find most confused classes
        confused_with = [(j, cm[i, j]) for j in range(len(class_names)) if j != i and cm[i, j] > 0]
        confused_with.sort(key=lambda x: x[1], reverse=True)
        
        print(f"\n  {true_class}:")
        print(f"    Correctly classified: {correct}/{total} ({correct/total*100:.1f}%)")
        if confused_with:
            print(f"    Most confused with: {class_names[confused_with[0][0]]} ({confused_with[0][1]} cases)")

plot_confusion_matrix_detailed()

## 5. Wrong Prediction Analysis

In [ ]:
def analyze_wrong_predictions():
    """Deep analysis of wrong predictions"""
    print("=" * 60)
    print("WRONG PREDICTION ANALYSIS")
    print("=" * 60)
    
    # Find wrong predictions
    wrong_indices = np.where(predictions != true_labels)[0]
    print(f"\n❌ Total wrong predictions: {len(wrong_indices)} ({len(wrong_indices)/len(predictions)*100:.1f}%)")
    
    # Analyze confusion patterns
    confusion_patterns = {}
    for idx in wrong_indices:
        true_label = true_labels[idx]
        pred_label = predictions[idx]
        pattern = f"{class_names[true_label]} → {class_names[pred_label]}"
        confusion_patterns[pattern] = confusion_patterns.get(pattern, 0) + 1
    
    print(f"\n🔄 Top Confusion Patterns:")
    sorted_patterns = sorted(confusion_patterns.items(), key=lambda x: x[1], reverse=True)
    for pattern, count in sorted_patterns[:10]:
        print(f"  {pattern}: {count} cases")
    
    # Analyze confidence scores for wrong predictions
    wrong_confidences = []
    correct_confidences = []
    
    for i in range(len(predictions)):
        confidence = np.max(probabilities[i])
        if predictions[i] != true_labels[i]:
            wrong_confidences.append(confidence)
        else:
            correct_confidences.append(confidence)
    
    print(f"\n📊 Confidence Analysis:")
    print(f"  Wrong predictions - Mean confidence: {np.mean(wrong_confidences):.3f}")
    print(f"  Correct predictions - Mean confidence: {np.mean(correct_confidences):.3f}")
    print(f"  High confidence wrong predictions (>0.9): {np.sum(np.array(wrong_confidences) > 0.9)}")
    
    return wrong_indices, confusion_patterns

wrong_indices, confusion_patterns = analyze_wrong_predictions()

In [ ]:
def visualize_wrong_predictions():
    """Visualize wrong prediction patterns"""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Confidence distribution for wrong vs correct
    ax1 = axes[0, 0]
    wrong_confidences = [np.max(probabilities[i]) for i in wrong_indices]
    correct_confidences = [np.max(probabilities[i]) for i in range(len(predictions)) 
                          if i not in wrong_indices]
    
    ax1.hist(correct_confidences, bins=30, alpha=0.7, label='Correct', color='green', density=True)
    ax1.hist(wrong_confidences, bins=30, alpha=0.7, label='Wrong', color='red', density=True)
    ax1.set_xlabel('Prediction Confidence')
    ax1.set_ylabel('Density')
    ax1.set_title('Confidence Distribution: Correct vs Wrong Predictions', fontweight='bold')
    ax1.legend()
    
    # 2. Top confusion patterns
    ax2 = axes[0, 1]
    top_patterns = sorted(confusion_patterns.items(), key=lambda x: x[1], reverse=True)[:10]
    patterns, counts = zip(*top_patterns)
    
    bars = ax2.barh(range(len(patterns)), counts)
    ax2.set_yticks(range(len(patterns)))
    ax2.set_yticklabels(patterns)
    ax2.set_xlabel('Number of Cases')
    ax2.set_title('Top Confusion Patterns', fontweight='bold')
    
    # Add value labels
    for bar, count in zip(bars, counts):
        width = bar.get_width()
        ax2.text(width + max(counts)*0.01, bar.get_y() + bar.get_height()/2,
                f'{count}', ha='left', va='center')
    
    # 3. Per-class error rates
    ax3 = axes[1, 0]
    error_rates = []
    for i, class_name in enumerate(class_names):
        class_indices = np.where(true_labels == i)[0]
        class_wrong = len([idx for idx in class_indices if idx in wrong_indices])
        error_rate = class_wrong / len(class_indices)
        error_rates.append(error_rate)
    
    bars = ax3.bar(class_names, error_rates, color='coral', alpha=0.7)
    ax3.set_xlabel('Medical Classes')
    ax3.set_ylabel('Error Rate')
    ax3.set_title('Per-Class Error Rates', fontweight='bold')
    ax3.tick_params(axis='x', rotation=45)
    
    # Add percentage labels
    for bar, rate in zip(bars, error_rates):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + max(error_rates)*0.01,
                f'{rate*100:.1f}%', ha='center', va='bottom')
    
    # 4. Sample wrong predictions
    ax4 = axes[1, 1]
    sample_wrong = np.random.choice(wrong_indices, min(9, len(wrong_indices)), replace=False)
    
    for i, idx in enumerate(sample_wrong):
        if i >= 9:
            break
            
        img = Image.open(test_paths[idx])
        img = img.resize((64, 64))
        
        row = i // 3
        col = i % 3
        
        ax4.imshow(img, cmap='gray', extent=[col, col+1, 3-row-1, 3-row])
        
        true_label = class_names[true_labels[idx]]
        pred_label = class_names[predictions[idx]]
        confidence = np.max(probabilities[idx])
        
        ax4.text(col + 0.5, 3-row-0.8, f'True: {true_label}', 
                ha='center', va='top', fontsize=8, color='green')
        ax4.text(col + 0.5, 3-row-0.2, f'Pred: {pred_label}', 
                ha='center', va='bottom', fontsize=8, color='red')
        ax4.text(col + 0.5, 3-row-0.5, f'{confidence:.2f}', 
                ha='center', va='center', fontsize=8, fontweight='bold')
    
    ax4.set_xlim(0, 3)
    ax4.set_ylim(0, 3)
    ax4.set_title('Sample Wrong Predictions\n(True/Pred/Confidence)', fontweight='bold')
    ax4.axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_wrong_predictions()

## 6. Grad-CAM Visualization

In [ ]:
class GradCAM:
    """Grad-CAM implementation for model interpretability"""
    def __init__(self, model, target_layer_name):
        self.model = model
        self.target_layer_name = target_layer_name
        self.gradients = None
        self.activations = None
        
        # Register hooks
        self._register_hooks()
    
    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output
        
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]
        
        # Find target layer
        target_layer = None
        for name, module in self.model.named_modules():
            if name == self.target_layer_name:
                target_layer = module
                break
        
        if target_layer is not None:
            target_layer.register_forward_hook(forward_hook)
            target_layer.register_backward_hook(backward_hook)
    
    def generate_cam(self, input_tensor, class_idx):
        self.model.eval()
        
        # Forward pass
        output = self.model(input_tensor)
        
        # Zero gradients
        self.model.zero_grad()
        
        # Backward pass for target class
        class_score = output[0, class_idx]
        class_score.backward()
        
        # Generate CAM
        gradients = self.gradients[0]  # [C, H, W]
        activations = self.activations[0]  # [C, H, W]
        
        # Global average pooling of gradients
        weights = torch.mean(gradients, dim=(1, 2))  # [C]
        
        # Weighted combination of activation maps
        cam = torch.sum(weights[:, None, None] * activations, dim=0)  # [H, W]
        
        # ReLU and normalize
        cam = F.relu(cam)
        cam = cam - cam.min()
        cam = cam / cam.max()
        
        return cam.cpu().numpy()

def create_gradcam_visualization(model, image_path, true_label, predicted_label, confidence):
    """Create Grad-CAM visualization for a single image"""
    # Load and preprocess image
    img = Image.open(image_path).convert('L')
    original_img = img.resize((256, 256))
    
    # Preprocess for model
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
    
    input_tensor = transform(img).unsqueeze(0).to(config.device)
    
    # Initialize Grad-CAM
    # Try different layer names based on model architecture
    layer_names = ['features.14', 'features.13', 'features.12', 'classifier.0']
    grad_cam = None
    
    for layer_name in layer_names:
        try:
            grad_cam = GradCAM(model, layer_name)
            cam = grad_cam.generate_cam(input_tensor, predicted_label)
            break
        except:
            continue
    
    if grad_cam is None:
        print(f"Could not create Grad-CAM for image {image_path}")
        return None
    
    # Resize CAM to match image
    cam_resized = cv2.resize(cam, (256, 256))
    
    # Create visualization
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original image
    ax1.imshow(original_img, cmap='gray')
    ax1.set_title(f'Original Image\nTrue: {true_label}\nPred: {predicted_label}\nConf: {confidence:.3f}', 
                 fontweight='bold')
    ax1.axis('off')
    
    # Grad-CAM heatmap
    heatmap = ax2.imshow(cam_resized, cmap='jet', alpha=0.8)
    ax2.set_title('Grad-CAM Heatmap', fontweight='bold')
    ax2.axis('off')
    plt.colorbar(heatmap, ax=ax2, fraction=0.046, pad=0.04)
    
    # Overlay
    ax3.imshow(original_img, cmap='gray')
    ax3.imshow(cam_resized, cmap='jet', alpha=0.4)
    ax3.set_title('Overlay', fontweight='bold')
    ax3.axis('off')
    
    plt.tight_layout()
    return fig

print("Grad-CAM implementation ready!")

In [ ]:
def analyze_gradcam_samples():
    """Generate Grad-CAM visualizations for interesting cases"""
    print("=" * 60)
    print("GRAD-CAM ANALYSIS")
    print("=" * 60)
    
    # Analyze different types of predictions
    cases_to_analyze = {
        'correct_high_conf': [],
        'correct_low_conf': [],
        'wrong_high_conf': [],
        'wrong_low_conf': []
    }
    
    for i in range(len(predictions)):
        confidence = np.max(probabilities[i])
        is_correct = predictions[i] == true_labels[i]
        
        if is_correct and confidence > 0.9:
            cases_to_analyze['correct_high_conf'].append(i)
        elif is_correct and confidence < 0.7:
            cases_to_analyze['correct_low_conf'].append(i)
        elif not is_correct and confidence > 0.9:
            cases_to_analyze['wrong_high_conf'].append(i)
        elif not is_correct and confidence < 0.7:
            cases_to_analyze['wrong_low_conf'].append(i)
    
    print(f"\n📊 Case Distribution for Grad-CAM Analysis:")
    for case_type, indices in cases_to_analyze.items():
        print(f"  {case_type}: {len(indices)} cases")
    
    # Generate visualizations for each case type
    model = trained_models[0]  # Use first model for Grad-CAM
    
    for case_type, indices in cases_to_analyze.items():
        if len(indices) == 0:
            print(f"\n⚠️  No {case_type} cases to analyze")
            continue
        
        print(f"\n🔍 Analyzing {case_type} (showing {min(3, len(indices))} examples):")
        
        sample_indices = np.random.choice(indices, min(3, len(indices)), replace=False)
        
        for idx in sample_indices:
            true_label = class_names[true_labels[idx]]
            pred_label = class_names[predictions[idx]]
            confidence = np.max(probabilities[idx])
            
            fig = create_gradcam_visualization(
                model, test_paths[idx], true_label, pred_label, confidence
            )
            
            if fig is not None:
                plt.show()
                plt.close(fig)
            
            print(f"  Image: {Path(test_paths[idx]).name}")
            print(f"  True: {true_label}, Predicted: {pred_label}, Confidence: {confidence:.3f}")

analyze_gradcam_samples()

## 7. Medical-Specific Analysis

In [ ]:
def medical_specific_analysis():
    """Medical domain-specific analysis"""
    print("=" * 60)
    print("MEDICAL-SPECIFIC ANALYSIS")
    print("=" * 60)
    
    # Medical imaging modality analysis
    print(f"\n🏥 Medical Imaging Modalities:")
    modality_info = {
        'AbdomenCT': 'Computed Tomography - Abdominal scans',
        'BreastMRI': 'Magnetic Resonance Imaging - Breast tissue',
        'CXR': 'X-Ray - Chest radiography',
        'ChestCT': 'Computed Tomography - Chest scans',
        'Hand': 'X-Ray - Hand radiography',
        'HeadCT': 'Computed Tomography - Head scans'
    }
    
    for class_name, description in modality_info.items():
        class_idx = class_names.index(class_name)
        class_accuracy = np.mean((predictions == true_labels) & (true_labels == class_idx))
        class_total = np.sum(true_labels == class_idx)
        class_correct = np.sum((predictions == true_labels) & (true_labels == class_idx))
        
        print(f"  {class_name:12}: {class_correct}/{class_total} ({class_accuracy*100:.1f}%)")
        print(f"    {description}")
    
    # Imaging technology performance
    print(f"\n📡 Imaging Technology Performance:")
    tech_performance = {
        'CT': [],
        'MRI': [],
        'X-Ray': []
    }
    
    for class_name, description in modality_info.items():
        class_idx = class_names.index(class_name)
        class_accuracy = np.mean((predictions == true_labels) & (true_labels == class_idx))
        
        if 'CT' in class_name:
            tech_performance['CT'].append(class_accuracy)
        elif 'MRI' in class_name:
            tech_performance['MRI'].append(class_accuracy)
        elif 'X-Ray' in class_name or class_name == 'Hand':
            tech_performance['X-Ray'].append(class_accuracy)
    
    for tech, accuracies in tech_performance.items():
        if accuracies:
            avg_acc = np.mean(accuracies)
            print(f"  {tech:6}: {avg_acc*100:.1f}% average accuracy")
    
    # Clinical implications
    print(f"\n⚕️  Clinical Implications:")
    
    # High-confidence wrong predictions (potentially dangerous)
    high_conf_wrong = []
    for i in wrong_indices:
        if np.max(probabilities[i]) > 0.95:
            high_conf_wrong.append(i)
    
    print(f"  High-confidence wrong predictions: {len(high_conf_wrong)}")
    print(f"  These could be clinically significant misdiagnoses")
    
    if high_conf_wrong:
        print(f"  Examples:")
        for idx in high_conf_wrong[:3]:
            true_label = class_names[true_labels[idx]]
            pred_label = class_names[predictions[idx]]
            confidence = np.max(probabilities[idx])
            print(f"    {true_label} → {pred_label} (confidence: {confidence:.3f})")
    
    return tech_performance, high_conf_wrong

tech_performance, high_conf_wrong = medical_specific_analysis()

In [ ]:
def visualize_medical_analysis():
    """Visualize medical-specific analysis"""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Modality performance comparison
    ax1 = axes[0, 0]
    modality_accuracies = []
    for class_name in class_names:
        class_idx = class_names.index(class_name)
        class_accuracy = np.mean((predictions == true_labels) & (true_labels == class_idx))
        modality_accuracies.append(class_accuracy)
    
    bars = ax1.bar(class_names, modality_accuracies, color=sns.color_palette("husl", len(class_names)))
    ax1.set_title('Performance by Medical Modality', fontweight='bold')
    ax1.set_ylabel('Accuracy')
    ax1.tick_params(axis='x', rotation=45)
    ax1.set_ylim(0, 1)
    
    # Add accuracy labels
    for bar, acc in zip(bars, modality_accuracies):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{acc*100:.1f}%', ha='center', va='bottom')
    
    # 2. Imaging technology comparison
    ax2 = axes[0, 1]
    tech_names = []
    tech_accs = []
    
    for tech, accuracies in tech_performance.items():
        if accuracies:
            tech_names.append(tech)
            tech_accs.append(np.mean(accuracies))
    
    bars = ax2.bar(tech_names, tech_accs, color=['skyblue', 'lightgreen', 'salmon'])
    ax2.set_title('Performance by Imaging Technology', fontweight='bold')
    ax2.set_ylabel('Average Accuracy')
    ax2.set_ylim(0, 1)
    
    # Add accuracy labels
    for bar, acc in zip(bars, tech_accs):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{acc*100:.1f}%', ha='center', va='bottom')
    
    # 3. Confidence vs Accuracy relationship
    ax3 = axes[1, 0]
    confidence_bins = np.linspace(0, 1, 11)
    accuracy_by_confidence = []
    
    for i in range(len(confidence_bins)-1):
        bin_mask = (np.max(probabilities, axis=1) >= confidence_bins[i]) & \
                   (np.max(probabilities, axis=1) < confidence_bins[i+1])
        
        if np.sum(bin_mask) > 0:
            bin_accuracy = np.mean(predictions[bin_mask] == true_labels[bin_mask])
            accuracy_by_confidence.append(bin_accuracy)
        else:
            accuracy_by_confidence.append(0)
    
    bin_centers = (confidence_bins[:-1] + confidence_bins[1:]) / 2
    ax3.plot(bin_centers, accuracy_by_confidence, 'bo-', linewidth=2, markersize=8)
    ax3.set_xlabel('Prediction Confidence')
    ax3.set_ylabel('Actual Accuracy')
    ax3.set_title('Confidence Calibration', fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Perfect Calibration')
    ax3.legend()
    
    # 4. Critical errors analysis
    ax4 = axes[1, 1]
    critical_errors = []
    normal_errors = []
    
    for i in wrong_indices:
        confidence = np.max(probabilities[i])
        if confidence > 0.9:
            critical_errors.append(i)
        else:
            normal_errors.append(i)
    
    labels = ['Critical Errors\n(>90% confidence)', 'Normal Errors\n(<90% confidence)']
    sizes = [len(critical_errors), len(normal_errors)]
    colors = ['red', 'orange']
    
    if sum(sizes) > 0:
        wedges, texts, autotexts = ax4.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
                                          startangle=90)
        ax4.set_title('Error Severity Distribution', fontweight='bold')
    else:
        ax4.text(0.5, 0.5, 'No errors to analyze', ha='center', va='center', 
                transform=ax4.transAxes, fontsize=12)
        ax4.set_title('Error Severity Distribution', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

visualize_medical_analysis()

## 8. Advanced Analysis: t-SNE and Feature Space

In [ ]:
def extract_features(model, data_loader, layer_name='classifier.0'):
    """Extract features from a specific layer"""
    features = []
    labels_list = []
    
    # Hook to capture features
    captured_features = []
    
    def hook_fn(module, input, output):
        captured_features.append(output.detach().cpu().numpy())
    
    # Register hook
    for name, module in model.named_modules():
        if name == layer_name:
            module.register_forward_hook(hook_fn)
            break
    
    model.eval()
    with torch.no_grad():
        for batch_idx, (images, batch_labels) in enumerate(data_loader):
            images = images.to(config.device)
            _ = model(images)  # Forward pass to trigger hook
            labels_list.extend(batch_labels.numpy())
            
            # Limit to avoid memory issues
            if batch_idx >= 50:  # Process ~50 batches
                break
    
    if captured_features:
        features = np.vstack(captured_features)
        return features, np.array(labels_list)
    else:
        return None, None

def perform_tsne_analysis():
    """Perform t-SNE analysis on learned features"""
    print("=" * 60)
    print("FEATURE SPACE ANALYSIS (t-SNE)")
    print("=" * 60)
    
    # Extract features
    print("\n🔍 Extracting features from trained model...")
    model = trained_models[0]
    features, feature_labels = extract_features(model, test_loader)
    
    if features is None:
        print("❌ Could not extract features")
        return
    
    print(f"Extracted {features.shape[0]} samples with {features.shape[1]} features")
    
    # Perform t-SNE
    print("\n📊 Performing t-SNE dimensionality reduction...")
    tsne = TSNE(n_components=2, random_state=config.seed, perplexity=30, n_iter=1000)
    tsne_features = tsne.fit_transform(features)
    
    # Visualize
    plt.figure(figsize=(12, 8))
    
    for i, class_name in enumerate(class_names):
        mask = feature_labels == i
        plt.scatter(tsne_features[mask, 0], tsne_features[mask, 1], 
                   label=class_name, alpha=0.7, s=30)
    
    plt.title('t-SNE Visualization of Learned Features', fontsize=16, fontweight='bold')
    plt.xlabel('t-SNE Component 1')
    plt.ylabel('t-SNE Component 2')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Analyze clustering quality
    print(f"\n📈 Clustering Analysis:")
    from sklearn.metrics import silhouette_score
    
    silhouette_avg = silhouette_score(tsne_features, feature_labels)
    print(f"  Silhouette score: {silhouette_avg:.3f}")
    
    if silhouette_avg > 0.5:
        print(f"  ✅ Good clustering - classes are well separated")
    elif silhouette_avg > 0.25:
        print(f"  ⚠️  Moderate clustering - some class overlap")
    else:
        print(f"  ❌ Poor clustering - significant class overlap")
    
    return tsne_features

tsne_results = perform_tsne_analysis()

## 9. Summary and Recommendations

In [ ]:
def generate_summary_report():
    """Generate comprehensive summary report"""
    print("=" * 80)
    print("MEDICAL MNIST - COMPREHENSIVE ANALYSIS SUMMARY")
    print("=" * 80)
    
    # Overall performance
    overall_accuracy = np.mean(predictions == true_labels)
    print(f"\n🎯 OVERALL PERFORMANCE:")
    print(f"  Test Accuracy: {overall_accuracy*100:.2f}%")
    print(f"  Total Test Samples: {len(predictions)}")
    print(f"  Correct Predictions: {np.sum(predictions == true_labels)}")
    print(f"  Wrong Predictions: {len(wrong_indices)}")
    
    # Data quality assessment
    print(f"\n📊 DATA QUALITY:")
    print(f"  Dataset Balance: {'Good' if imbalance_ratio < 1.5 else 'Moderate' if imbalance_ratio < 2.0 else 'Poor'}")
    print(f"  Imbalance Ratio: {imbalance_ratio:.2f}:1")
    print(f"  Total Images: {len(image_paths)}")
    print(f"  Classes: {len(class_names)}")
    
    # Model reliability
    print(f"\n🔍 MODEL RELIABILITY:")
    
    # Confidence calibration
    confidences = np.max(probabilities, axis=1)
    avg_confidence = np.mean(confidences)
    confidence_accuracy_correlation = np.corrcoef(confidences, predictions == true_labels)[0, 1]
    
    print(f"  Average Confidence: {avg_confidence:.3f}")
    print(f"  Confidence-Accuracy Correlation: {confidence_accuracy_correlation:.3f}")
    
    if confidence_accuracy_correlation > 0.7:
        print(f"  ✅ Well-calibrated model")
    elif confidence_accuracy_correlation > 0.3:
        print(f"  ⚠️  Moderately calibrated")
    else:
        print(f"  ❌ Poorly calibrated - confidence doesn't match accuracy")
    
    # Critical errors
    critical_errors = len([i for i in wrong_indices if np.max(probabilities[i]) > 0.9])
    print(f"  Critical Errors (>90% confidence): {critical_errors}")
    
    # Per-class performance
    print(f"\n🏥 PER-CLASS PERFORMANCE:")
    worst_classes = []
    best_classes = []
    
    for i, class_name in enumerate(class_names):
        class_mask = true_labels == i
        if np.sum(class_mask) > 0:
            class_accuracy = np.mean(predictions[class_mask] == true_labels[class_mask])
            print(f"  {class_name:12}: {class_accuracy*100:.2f}%")
            
            if class_accuracy < 0.95:
                worst_classes.append((class_name, class_accuracy))
            if class_accuracy > 0.99:
                best_classes.append((class_name, class_accuracy))
    
    # Recommendations
    print(f"\n💡 RECOMMENDATIONS:")
    
    if imbalance_ratio > 1.5:
        print(f"  🔧 Consider class weighting or data augmentation for imbalanced classes")
    
    if critical_errors > 10:
        print(f"  ⚠️  High number of critical errors - review model confidence calibration")
        print(f"      Consider temperature scaling or Platt scaling")
    
    if worst_classes:
        print(f"  🎯 Focus on improving performance for: {[cls for cls, _ in worst_classes]}")
        print(f"      Consider class-specific data augmentation or ensemble methods")
    
    if confidence_accuracy_correlation < 0.5:
        print(f"  📊 Improve confidence calibration using:")
        print(f"      - Temperature scaling")
        print(f"      - Label smoothing")
        print(f"      - Ensemble methods")
    
    print(f"  🏥 Medical-specific considerations:")
    print(f"      - Implement uncertainty quantification for clinical settings")
    print(f"      - Consider cost-sensitive learning for different misdiagnosis types")
    print(f"      - Add attention mechanisms for better interpretability")
    
    return {
        'accuracy': overall_accuracy,
        'imbalance_ratio': imbalance_ratio,
        'critical_errors': critical_errors,
        'confidence_correlation': confidence_accuracy_correlation,
        'worst_classes': worst_classes,
        'best_classes': best_classes
    }

summary_results = generate_summary_report()

In [ ]:
def save_analysis_results():
    """Save analysis results to files"""
    print("\n💾 Saving analysis results...")
    
    # Create results directory
    results_dir = Paths.OUTPUTS_DIR / "analysis_results"
    results_dir.mkdir(exist_ok=True)
    
    # Save detailed predictions
    results_df = pd.DataFrame({
        'image_path': [Path(p).name for p in test_paths],
        'true_label': [class_names[i] for i in true_labels],
        'predicted_label': [class_names[i] for i in predictions],
        'confidence': np.max(probabilities, axis=1),
        'is_correct': predictions == true_labels
    })
    
    results_df.to_csv(results_dir / 'detailed_predictions.csv', index=False)
    print(f"  Saved detailed predictions to {results_dir / 'detailed_predictions.csv'}")
    
    # Save wrong predictions analysis
    wrong_df = results_df[~results_df['is_correct']].copy()
    wrong_df.to_csv(results_dir / 'wrong_predictions.csv', index=False)
    print(f"  Saved wrong predictions to {results_dir / 'wrong_predictions.csv'}")
    
    # Save summary statistics
    summary_stats = {
        'total_samples': len(predictions),
        'accuracy': float(np.mean(predictions == true_labels)),
        'total_wrong': int(len(wrong_indices)),
        'critical_errors': int(len([i for i in wrong_indices if np.max(probabilities[i]) > 0.9])),
        'class_distribution': {class_names[i]: int(np.sum(true_labels == i)) for i in range(len(class_names))},
        'per_class_accuracy': {class_names[i]: float(np.mean((predictions == true_labels) & (true_labels == i))) / max(1, np.sum(true_labels == i)) for i in range(len(class_names))}
    }
    
    import json
    with open(results_dir / 'summary_statistics.json', 'w') as f:
        json.dump(summary_stats, f, indent=2)
    
    print(f"  Saved summary statistics to {results_dir / 'summary_statistics.json'}")
    print(f"\n✅ All analysis results saved!")

save_analysis_results()

print("\n" + "=" * 80)
print("🎉 MEDICAL MNIST DEBUGGING & ANALYSIS COMPLETE!")
print("=" * 80)
print("\n📁 Check the analysis_results directory for detailed reports.")
print("🔍 Key insights and recommendations have been provided above.")
print("🏥 Remember: This is medical imaging - always consider clinical implications!")